# Load All Task Run Reports

This notebook loads every report matching:

`logs/<model name>/<run name>/task_run_report.json`

It creates reusable pandas DataFrames for run-level summaries, task-level results, and per-tool appropriateness scores. Use this as the first notebook when you want to compare multiple models or inspect all experiment outputs together.

## How To Use

1. Select the repo `.venv` kernel in VS Code/Jupyter.
2. Run the import/config cells.
3. Leave `MODEL_FILTER = None` to load every model folder under `logs/`.
4. Set `MODEL_FILTER` to a model folder name, or a list of folder names, to load only selected models.
5. Use `reports_df` for one row per run, `task_runs_df` for one row per task-run, and `tool_scores_df` for one row per scored tool call.

In [52]:
from __future__ import annotations

import json
import re
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs pandas. Select the project .venv kernel, "
        "or install dependencies with: uv sync"
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Configuration

Change only this cell for most use cases. The notebook is written to work whether you run it from the repo root or from the `analysis/` folder.

In [53]:
# Set to None to load every model folder under logs/.
# Examples:
# MODEL_FILTER = "qwen3.6-plus"
# MODEL_FILTER = ["qwen3.6-plus", "gpt5.5"]
MODEL_FILTER = None

# Set True if you want CSV copies of the DataFrames written to analysis/outputs/.
EXPORT_CSV = False

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "analysis" else Path(".").resolve()
LOGS_ROOT = PROJECT_ROOT / "logs"
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "outputs"

assert LOGS_ROOT.exists(), f"Logs folder not found: {LOGS_ROOT}"

## Discover Reports

This scans only one level below each model folder, so it includes paths shaped like `logs/model/run/task_run_report.json` and skips deeper archival folders such as `logs/results/...`.

In [54]:
def normalise_model_filter(model_filter):
    if model_filter is None:
        return None
    if isinstance(model_filter, str):
        return {model_filter}
    return set(model_filter)


def discover_report_paths(logs_root: Path, model_filter=None) -> list[Path]:
    allowed_models = normalise_model_filter(model_filter)
    report_paths = []

    for model_dir in sorted(p for p in logs_root.iterdir() if p.is_dir()):
        if allowed_models is not None and model_dir.name not in allowed_models:
            continue

        for run_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
            report_path = run_dir / "task_run_report.json"
            if report_path.exists():
                report_paths.append(report_path)

    return report_paths


report_paths = discover_report_paths(LOGS_ROOT, MODEL_FILTER)
print(f"Found {len(report_paths)} reports")
for path in report_paths[:10]:
    print(path.relative_to(PROJECT_ROOT))

Found 18 reports
logs\claude-optus-4.6\1anthropic_claude-opus-4.6_20260512T082634Z\task_run_report.json
logs\claude-optus-4.6\2anthropic_claude-opus-4.6_20260512T084016Z\task_run_report.json
logs\claude-optus-4.6\3anthropic_claude-opus-4.6_20260516T090351Z\task_run_report.json
logs\claude-sonnet-4.6\1anthropic_claude-sonnet-4.6_20260512T072817Z\task_run_report.json
logs\claude-sonnet-4.6\2anthropic_claude-sonnet-4.6_20260512T073036Z\task_run_report.json
logs\claude-sonnet-4.6\3anthropic_claude-sonnet-4.6_20260512T073516Z\task_run_report.json
logs\gpt5.5\openai_gpt-5.5_20260512T075029Z\task_run_report.json
logs\gpt5.5\openai_gpt-5.5_20260512T075044Z\task_run_report.json
logs\gpt5.5\openai_gpt-5.5_20260512T075850Z\task_run_report.json
logs\kimi-k2.6\1moonshotai_kimi-k2.6_20260512T065725Z\task_run_report.json


## Load JSON Reports

`reports` keeps the raw JSON objects in memory. The following cells flatten those objects into DataFrames while preserving the original model folder, run folder, and report path for traceability.

In [55]:
def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def leading_run_index(run_name: str):
    match = re.match(r"^(\d+)", run_name)
    return int(match.group(1)) if match else None


reports = []
for report_path in report_paths:
    run_dir = report_path.parent
    model_dir = run_dir.parent
    report = read_json(report_path)
    reports.append({
        "model_name": model_dir.name,
        "run_name": run_dir.name,
        "run_index": leading_run_index(run_dir.name),
        "report_path": str(report_path.relative_to(PROJECT_ROOT)),
        "report": report,
    })

len(reports)

18

## Build DataFrames

- `reports_df`: one row per `task_run_report.json`, mostly run-level summary fields.
- `task_runs_df`: one row per task result inside each report.
- `tool_scores_df`: one row per scored tool call from the `tool_appropriateness` section, when present.

In [56]:
report_rows = []
task_rows = []
tool_score_rows = []

for item in reports:
    report = item["report"]
    summary = report.get("summary", {}) or {}
    base = {
        "model_name": item["model_name"],
        "run_name": item["run_name"],
        "run_index": item["run_index"],
        "report_path": item["report_path"],
        "generated_at": report.get("generated_at"),
        "source_csv": report.get("source_csv"),
    }

    report_rows.append({**base, **summary})

    for result in report.get("results", []) or []:
        validation = result.get("ground_truth_validation") or {}
        query_trace = result.get("query_trace") or {}
        tool_appropriateness = result.get("tool_appropriateness") or {}

        task_base = {
            **base,
            "task_id": result.get("task_id"),
            "prompt": result.get("prompt"),
            "status": result.get("status"),
            "answer_type": result.get("answer_type"),
            "passed": validation.get("passed", result.get("passed")),
            "ground_truth_score": validation.get("score"),
            "validation_strategy": validation.get("strategy"),
            "parameter_schema_valid_rate": result.get("parameter_schema_valid_rate"),
            "constraint_compliance_rate": result.get("constraint_compliance_rate"),
            "tool_appropriateness": tool_appropriateness.get("average_score"),
            "tool_appropriateness_weighted": tool_appropriateness.get("weighted_score"),
            "total_tokens": result.get("total_tokens"),
            "input_tokens": query_trace.get("input_tokens"),
            "output_tokens": query_trace.get("output_tokens"),
            "total_steps": result.get("total_steps"),
            "total_tool_calls": result.get("total_tool_calls"),
            "latency": result.get("latency"),
            "end_to_end_latency": result.get("end_to_end_latency"),
            "attempt_count": result.get("attempt_count"),
            "tools_used": result.get("tools_used") or query_trace.get("tools_used"),
            "final_answer": query_trace.get("final_answer", result.get("final_answer")),
        }
        task_rows.append(task_base)

        for call_index, scored_call in enumerate(tool_appropriateness.get("scored_calls") or [], start=1):
            tool_score_rows.append({
                **base,
                "task_id": result.get("task_id"),
                "call_index": call_index,
                **scored_call,
            })

reports_df = pd.DataFrame(report_rows)
task_runs_df = pd.DataFrame(task_rows)
tool_scores_df = pd.DataFrame(tool_score_rows)

if not task_runs_df.empty:
    task_runs_df["passed"] = task_runs_df["passed"].astype("boolean")
    numeric_columns = [
        "ground_truth_score",
        "parameter_schema_valid_rate",
        "constraint_compliance_rate",
        "tool_appropriateness",
        "tool_appropriateness_weighted",
        "total_tokens",
        "input_tokens",
        "output_tokens",
        "total_steps",
        "total_tool_calls",
        "latency",
        "end_to_end_latency",
        "attempt_count",
    ]
    for col in numeric_columns:
        if col in task_runs_df.columns:
            task_runs_df[col] = pd.to_numeric(task_runs_df[col], errors="coerce")

if not tool_scores_df.empty and "score" in tool_scores_df.columns:
    tool_scores_df["score"] = pd.to_numeric(tool_scores_df["score"], errors="coerce")

print(f"reports_df: {reports_df.shape[0]} rows x {reports_df.shape[1]} columns")
print(f"task_runs_df: {task_runs_df.shape[0]} rows x {task_runs_df.shape[1]} columns")
print(f"tool_scores_df: {tool_scores_df.shape[0]} rows x {tool_scores_df.shape[1]} columns")

reports_df: 18 rows x 28 columns
task_runs_df: 630 rows x 27 columns
tool_scores_df: 3106 rows x 12 columns


## Quick Checks

These lightweight checks confirm which models and runs were loaded. If a model is missing, check that its reports are located exactly one folder below its model folder.

In [57]:
display(
    reports_df[["model_name", "run_name", "run_index", "generated_at", "model", "total_tasks", "Task Completion Rate (TCR)"]]
    .sort_values(["model_name", "run_index", "run_name"])
)

display(task_runs_df.head())
display(tool_scores_df.head())

,model_name,run_name,run_index,generated_at,model,total_tasks,Task Completion Rate (TCR)
0,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,2026-05-12T08:31:32Z,anthropic/claude-opus-4.6,35,0.9714
1,claude-optus-4.6,2anthropic_claude-opus-4.6_20260512T084016Z,2.0000,2026-05-12T08:45:20Z,anthropic/claude-opus-4.6,35,0.9429
2,claude-optus-4.6,3anthropic_claude-opus-4.6_20260516T090351Z,3.0000,2026-05-16T09:08:58Z,anthropic/claude-opus-4.6,35,0.9143
3,claude-sonnet-4.6,1anthropic_claude-sonnet-4.6_20260512T072817Z,1.0000,2026-05-12T07:33:08Z,anthropic/claude-sonnet-4.6,35,0.9143
4,claude-sonnet-4.6,2anthropic_claude-sonnet-4.6_20260512T073036Z,2.0000,2026-05-12T07:35:32Z,anthropic/claude-sonnet-4.6,35,0.8857
5,claude-sonnet-4.6,3anthropic_claude-sonnet-4.6_20260512T073516Z,3.0000,2026-05-12T07:39:53Z,anthropic/claude-sonnet-4.6,35,0.8286
6,gpt5.5,openai_gpt-5.5_20260512T075029Z,NaN,2026-05-12T07:55:22Z,openai/gpt-5.5,35,0.8286
7,gpt5.5,openai_gpt-5.5_20260512T075044Z,NaN,2026-05-12T07:55:16Z,openai/gpt-5.5,35,0.8286
8,gpt5.5,openai_gpt-5.5_20260512T075850Z,NaN,2026-05-12T08:03:20Z,openai/gpt-5.5,35,0.8857
9,kimi-k2.6,1moonshotai_kimi-k2.6_20260512T065725Z,1.0000,2026-05-12T07:09:53Z,moonshotai/kimi-k2.6,35,0.8857


,model_name,run_name,run_index,report_path,generated_at,source_csv,task_id,prompt,status,answer_type,passed,ground_truth_score,validation_strategy,parameter_schema_valid_rate,constraint_compliance_rate,tool_appropriateness,tool_appropriateness_weighted,total_tokens,input_tokens,output_tokens,total_steps,total_tool_calls,latency,end_to_end_latency,attempt_count,tools_used,final_answer
0,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A1,how many posts does twitter collection have?,completed,scalar,True,1,scalar_deterministic,1.0000,NaN,1.0000,1.0000,"9,302.0000","9,183.0000",119.0000,2.0000,1.0000,8.9180,8.9312,1,[get_collection_summary],"{""status"":""success"",""answer"":{""value"":18587322..."
1,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A2,Compare the harvesting date span of all availa...,completed,text,True,1,text_judge,1.0000,NaN,1.0000,8.0000,"15,823.0000","14,653.0000","1,170.0000",3.0000,8.0000,21.4419,30.8165,1,"[get_collections, get_collection_summary, get_...",Now I have all the data. Let me compute the ha...
2,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A3,"Compare Reddit, Mastodon, and Twitter by total...",completed,text,True,1,text_judge,1.0000,NaN,1.0000,4.0000,"14,821.0000","14,220.0000",601.0000,3.0000,4.0000,15.4138,28.6152,1,"[get_collections, get_collection_summary, get_...",I now have all the data needed to provide a co...
3,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A4,List all available collections and their start...,completed,text,True,1,text_judge,1.0000,NaN,1.0000,8.0000,"15,193.0000","14,597.0000",596.0000,3.0000,8.0000,12.1675,23.4208,1,"[get_collections, get_collection_summary, get_...","{\n ""status"": ""success"",\n ""answer"": {\n ..."
4,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A5,Which available collections have harvesting da...,completed,text,True,1,text_judge,1.0000,NaN,1.0000,8.0000,"15,425.0000","14,647.0000",778.0000,3.0000,8.0000,15.4860,25.2906,1,"[get_collections, get_collection_summary, get_...",I now have all the data needed to answer the q...


,model_name,run_name,run_index,report_path,generated_at,source_csv,task_id,call_index,tool_name,category,appropriateness,score
0,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A1,1,get_collection_summary,core,core,1.0000
1,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A2,1,get_collections,core,core,1.0000
2,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A2,2,get_collection_summary,core,core,1.0000
3,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A2,3,get_collection_summary,core,core,1.0000
4,claude-optus-4.6,1anthropic_claude-opus-4.6_20260512T082634Z,1.0000,logs\claude-optus-4.6\1anthropic_claude-opus-4...,2026-05-12T08:31:32Z,task\taskset35.csv,A2,4,get_collection_summary,core,core,1.0000


## Example Aggregations

These are optional examples showing how to start comparing models. You can copy and modify these cells for your thesis tables.

In [58]:
model_summary = task_runs_df.groupby("model_name", as_index=False).agg(
    runs=("run_name", "nunique"),
    task_runs=("task_id", "count"),
    unique_tasks=("task_id", "nunique"),
    tcr=("passed", "mean"),
    average_psv=("parameter_schema_valid_rate", "mean"),
    average_ccr=("constraint_compliance_rate", "mean"),
    average_tool_appropriateness=("tool_appropriateness", "mean"),
    average_latency=("latency", "mean"),
    average_tokens=("total_tokens", "mean"),
)

display(model_summary.sort_values("model_name"))

,model_name,runs,task_runs,unique_tasks,tcr,average_psv,average_ccr,average_tool_appropriateness,average_latency,average_tokens
0,claude-optus-4.6,3,105,35,0.9429,1.0000,0.9174,0.8403,24.5356,"19,048.6381"
1,claude-sonnet-4.6,3,105,35,0.8762,1.0000,0.9131,0.7930,23.3715,"21,040.1524"
2,gpt5.5,3,105,35,0.8476,0.9981,1.0000,0.9168,22.4569,"10,323.2190"
3,kimi-k2.6,3,105,35,0.8571,1.0000,0.9116,0.8466,82.1878,"22,033.7379"
4,llama4,3,105,35,0.0667,0.9861,0.7698,0.8147,20.7645,"10,227.3131"
5,qwen3.6-plus,3,105,35,0.8381,1.0000,0.9194,0.8349,52.0183,"23,077.0476"


## Mean TCR and Pass@3

This table reports the mean task completion rate across runs for each model and Pass@3, defined as the share of unique tasks that passed in at least one of the model's runs.

In [59]:
tcr_by_run = task_runs_df.groupby(["model_name", "run_name"], as_index=False).agg(
    task_runs=("task_id", "count"),
    successful_task_runs=("passed", "sum"),
    task_completion_rate=("passed", "mean"),
)

pass_at_3_by_task = task_runs_df.groupby(["model_name", "task_id"], as_index=False).agg(
    passed_at_least_once=("passed", "any"),
)

mean_tcr_pass3 = tcr_by_run.groupby("model_name", as_index=False).agg(
    #runs=("run_name", "nunique"),
    mean_task_completion_rate=("task_completion_rate", "mean"),
).merge(
    pass_at_3_by_task.groupby("model_name", as_index=False).agg(
        #unique_tasks=("task_id", "count"),
        tasks_passed_at_least_once=("passed_at_least_once", "sum"),
        pass_at_3=("passed_at_least_once", "mean"),
    ),
    on="model_name",
    how="left",
)

display(mean_tcr_pass3.sort_values("mean_task_completion_rate", ascending=False))

,model_name,mean_task_completion_rate,tasks_passed_at_least_once,pass_at_3
0,claude-optus-4.6,0.9429,34,0.9714
1,claude-sonnet-4.6,0.8762,34,0.9714
3,kimi-k2.6,0.8571,33,0.9429
2,gpt5.5,0.8476,34,0.9714
5,qwen3.6-plus,0.8381,33,0.9429
4,llama4,0.0667,4,0.1143


## Performance by Task Category

Task category is inferred from the first letter of `task_id`. For example, `A1` to `A7` are category `A`.

Category labels:

- **A**: Discovery & Dataset Overview
- **B**: Filtered Counting & Lookup
- **C**: Aggregation & Temporal Pattern Analysis
- **D**: Comparative Analysis
- **E**: Exploratory NLP Analysis

In [60]:
TASK_CATEGORY_LABELS = {
    "A": "Discovery & Dataset Overview",
    "B": "Filtered Counting & Lookup",
    "C": "Aggregation & Temporal Pattern Analysis",
    "D": "Comparative Analysis",
    "E": "Exploratory NLP Analysis",
}

task_runs_with_category = task_runs_df.copy()
task_runs_with_category["task_category"] = task_runs_with_category["task_id"].astype(str).str.extract(r"^([A-Za-z])", expand=False).str.upper()
task_runs_with_category["task_category_name"] = task_runs_with_category["task_category"].map(TASK_CATEGORY_LABELS).fillna("Unknown")

performance_by_task_category = task_runs_with_category.groupby(
    ["model_name", "task_category", "task_category_name"],
    as_index=False,
).agg(
    runs=("run_name", "nunique"),
    task_runs=("task_id", "count"),
    unique_tasks=("task_id", "nunique"),
    successful_task_runs=("passed", "sum"),
    task_completion_rate=("passed", "mean"),
    average_psv=("parameter_schema_valid_rate", "mean"),
    average_ccr=("constraint_compliance_rate", "mean"),
    average_tool_appropriateness=("tool_appropriateness", "mean"),
    average_latency=("latency", "mean"),
    average_tokens=("total_tokens", "mean"),
)

category_pass_at_3 = task_runs_with_category.groupby(
    ["model_name", "task_category", "task_id"],
    as_index=False,
).agg(passed_at_least_once=("passed", "any"))

category_pass_at_3 = category_pass_at_3.groupby(
    ["model_name", "task_category"],
    as_index=False,
).agg(pass_at_3=("passed_at_least_once", "mean"))

performance_by_task_category = performance_by_task_category.merge(
    category_pass_at_3,
    on=["model_name", "task_category"],
    how="left",
)

#display(performance_by_task_category.sort_values(["model_name", "task_category"]))

### Category Summary Across Models

This table averages across all loaded models to show which task categories are easiest or hardest overall.

In [61]:
category_summary = task_runs_with_category.groupby(
    ["task_category", "task_category_name"],
    as_index=False,
).agg(
    models=("model_name", "nunique"),
    task_runs=("task_id", "count"),
    unique_tasks=("task_id", "nunique"),
    successful_task_runs=("passed", "sum"),
    task_completion_rate=("passed", "mean"),
    average_psv=("parameter_schema_valid_rate", "mean"),
    average_ccr=("constraint_compliance_rate", "mean"),
    average_tool_appropriateness=("tool_appropriateness", "mean"),
    average_latency=("latency", "mean"),
    average_tokens=("total_tokens", "mean"),
)

# display(category_summary.sort_values("task_category"))

### Model TCR by Task Category

This pivot table compares each model's task completion rate across task categories. Rows are categories and columns are models.

In [62]:
model_tcr_by_category = performance_by_task_category.pivot_table(
    index=["task_category", "task_category_name"],
    columns="model_name",
    values="task_completion_rate",
)

model_tcr_by_category = model_tcr_by_category.sort_index()
model_tcr_by_category_percent = model_tcr_by_category.map(lambda value: f"{value:.2%}" if pd.notna(value) else "")
display(model_tcr_by_category_percent)

,model_name,claude-optus-4.6,claude-sonnet-4.6,gpt5.5,kimi-k2.6,llama4,qwen3.6-plus
task_category,task_category_name,,,,,,
A,Discovery & Dataset Overview,100.00%,100.00%,100.00%,100.00%,28.57%,100.00%
B,Filtered Counting & Lookup,100.00%,100.00%,90.48%,95.24%,4.76%,85.71%
C,Aggregation & Temporal Pattern Analysis,85.71%,71.43%,85.71%,80.95%,0.00%,71.43%
D,Comparative Analysis,95.24%,76.19%,71.43%,76.19%,0.00%,90.48%
E,Exploratory NLP Analysis,90.48%,90.48%,76.19%,76.19%,0.00%,71.43%


## Tool-Use Behaviour by Task Category

This section reports tool-use behaviour for each model and task category:

- **average_tool_calls**: average tool calls per task-run.
- **average_core_tool_calls**: average scored `core` tool calls per task-run.
- **TA**: average task-level tool appropriateness score.

In [ ]:
tool_calls_by_category = task_runs_with_category.groupby(
    ["model_name", "task_category", "task_category_name"],
    as_index=False,
).agg(
    task_runs=("task_id", "count"),
    total_tool_calls=("total_tool_calls", "sum"),
    average_tool_calls=("total_tool_calls", "mean"),
    ta=("tool_appropriateness", "mean"),
)

tool_scores_with_category = tool_scores_df.copy()
tool_scores_with_category["task_category"] = tool_scores_with_category["task_id"].astype(str).str.extract(r"^([A-Za-z])", expand=False).str.upper()
tool_scores_with_category["task_category_name"] = tool_scores_with_category["task_category"].map(TASK_CATEGORY_LABELS).fillna("Unknown")
tool_scores_with_category["is_core_tool_call"] = tool_scores_with_category["appropriateness"].astype(str).str.lower().eq("core")

core_tool_call_counts = tool_scores_with_category.groupby(
    ["model_name", "run_name", "task_id", "task_category", "task_category_name"],
    as_index=False,
).agg(
    scored_tool_calls=("call_index", "count"),
    core_tool_calls=("is_core_tool_call", "sum"),
)

task_category_keys = task_runs_with_category[["model_name", "run_name", "task_id", "task_category", "task_category_name"]].drop_duplicates()
core_tool_call_counts = task_category_keys.merge(
    core_tool_call_counts,
    on=["model_name", "run_name", "task_id", "task_category", "task_category_name"],
    how="left",
)
core_tool_call_counts[["scored_tool_calls", "core_tool_calls"]] = core_tool_call_counts[["scored_tool_calls", "core_tool_calls"]].fillna(0).astype(int)

core_tool_calls_by_category = core_tool_call_counts.groupby(
    ["model_name", "task_category", "task_category_name"],
    as_index=False,
).agg(
    total_scored_tool_calls=("scored_tool_calls", "sum"),
    total_core_tool_calls=("core_tool_calls", "sum"),
    average_core_tool_calls=("core_tool_calls", "mean"),
)

tool_use_by_category = tool_calls_by_category.merge(
    core_tool_calls_by_category,
    on=["model_name", "task_category", "task_category_name"],
    how="left",
)

tool_use_by_category[["total_scored_tool_calls", "total_core_tool_calls"]] = tool_use_by_category[["total_scored_tool_calls", "total_core_tool_calls"]].fillna(0).astype(int)
tool_use_by_category["average_core_tool_calls"] = tool_use_by_category["average_core_tool_calls"].fillna(0)
tool_use_by_category["core_tool_call_share"] = tool_use_by_category["total_core_tool_calls"] / tool_use_by_category["total_scored_tool_calls"].replace(0, pd.NA)

#display(tool_use_by_category.sort_values(["model_name", "task_category"]))

### Tool-Use Pivot Tables

These compact views make it easier to compare models across categories.

In [66]:
tool_calls_pivot = tool_use_by_category.pivot_table(
    index=["task_category", "task_category_name"],
    columns="model_name",
    values="average_tool_calls",
)

core_tool_calls_pivot = tool_use_by_category.pivot_table(
    index=["task_category", "task_category_name"],
    columns="model_name",
    values="average_core_tool_calls",
)

ta_pivot = tool_use_by_category.pivot_table(
    index=["task_category", "task_category_name"],
    columns="model_name",
    values="ta",
)

#display(tool_calls_pivot.sort_index())
#display(core_tool_calls_pivot.sort_index())
display(ta_pivot.sort_index().map(lambda value: f"{value:.3f}" if pd.notna(value) else ""))

,model_name,claude-optus-4.6,claude-sonnet-4.6,gpt5.5,kimi-k2.6,llama4,qwen3.6-plus
task_category,task_category_name,,,,,,
A,Discovery & Dataset Overview,1.000,1.000,1.000,1.000,1.000,1.000
B,Filtered Counting & Lookup,0.960,0.879,0.845,0.801,0.688,0.721
C,Aggregation & Temporal Pattern Analysis,0.752,0.710,0.936,0.793,0.567,0.849
D,Comparative Analysis,0.727,0.638,0.904,0.741,0.657,0.669
E,Exploratory NLP Analysis,0.762,0.738,0.899,0.893,0.975,0.937


## Optional Export

Set `EXPORT_CSV = True` in the configuration cell if you want these tables written to `analysis/outputs/`.

In [65]:
if EXPORT_CSV:
    OUTPUT_DIR.mkdir(exist_ok=True)
    reports_df.to_csv(OUTPUT_DIR / "all_reports_summary.csv", index=False)
    task_runs_df.to_csv(OUTPUT_DIR / "all_task_runs.csv", index=False)
    tool_scores_df.to_csv(OUTPUT_DIR / "all_tool_scores.csv", index=False)
    model_summary.to_csv(OUTPUT_DIR / "all_model_summary.csv", index=False)
    tool_use_by_category.to_csv(OUTPUT_DIR / "tool_use_by_category.csv", index=False)
    print(f"Exported CSVs to {OUTPUT_DIR}")
else:
    print("CSV export skipped. Set EXPORT_CSV = True to write output files.")

CSV export skipped. Set EXPORT_CSV = True to write output files.
